In [18]:
import os
import pandas as pd
from sklearn.linear_model import LinearRegression

from aq_edge.datautils.air_quality_analysis import (setup_plotting, load_station_data, show_data_overview, calculate_station_statistics, plot_time_series, plot_distributions, analyze_correlations, analyze_missing_data, analyze_outliers, generate_pdf_report, generate_executive_summary_pdf)

from datetime import datetime

In [76]:
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import plotly.graph_objects as go

MAX_SHORT: int = 2   # 2 samples x 15 min = 30 min
MAX_MEDIUM: int = 4  # 4 samples x 15 min = 1 h
WINDOW: int = 16      # valid samples used on each side of a gap

def detect_outliers_iqr(
    df: pd.DataFrame, factor: float = 1.5
) -> pd.DataFrame:
    """Flag values outside ``factor`` x IQR beyond Q1 and Q3."""
    q1, q3 = df.quantile(0.25), df.quantile(0.75)
    iqr = q3 - q1
    return (df < q1 - factor * iqr) | (df > q3 + factor * iqr)

def fill_short_gaps(
    df: pd.DataFrame, max_len: int = MAX_SHORT
) -> pd.DataFrame:
    """Interpolate runs of at most ``max_len`` consecutive NaNs."""
    filled = df.copy()
    for col in df.columns:
        is_nan = df[col].isna()
        run_id = (is_nan != is_nan.shift()).cumsum()
        run_len = is_nan.groupby(run_id).transform("sum")
        interpolated = df[col].interpolate(
            method="time", limit_area="inside"
        )
        short = is_nan & (run_len <= max_len)
        filled[col] = df[col].where(~short, interpolated)
    return filled

def fill_medium_gaps(
    df: pd.DataFrame,
    min_len: int = MAX_SHORT,
    max_len: int = MAX_MEDIUM,
    window: int = WINDOW,
) -> pd.DataFrame:
    """Fill NaN runs of ``min_len`` < length <= ``max_len`` samples.

    A ``LinearRegression`` on time is fitted to up to ``window`` valid
    samples on each side of the gap. Gaps at the start or end of the
    series are left untouched.
    """
    filled = df.copy()
    hours = pd.Series(
        (df.index - df.index[0]) / pd.Timedelta("1h"),
        index=df.index,
        name="hours",
    )

    for col in df.columns:
        series = df[col]
        is_nan = series.isna()
        run_id = (is_nan != is_nan.shift()).cumsum()
        run_len = is_nan.groupby(run_id).transform("sum")
        medium = is_nan & (run_len > min_len) & (run_len <= max_len)

        for _, gap in series[medium].groupby(run_id[medium]):
            before = series[: gap.index[0]].dropna().iloc[-window:]
            after = series[gap.index[-1]:].dropna().iloc[:window]
            if before.empty or after.empty:
                continue

            train = pd.concat([before, after])
            model = LinearRegression().fit(
                hours.loc[train.index].to_frame(), train
            )
            filled.loc[gap.index, col] = model.predict(
                hours.loc[gap.index].to_frame()
            )
    return filled

def plot_stages_plotly(
    raw: pd.DataFrame,
    short: pd.DataFrame,
    medium: pd.DataFrame,
    station: str,
    col: str = "VOC",
    start: str | None = None,
    end: str | None = None,
) -> go.Figure:
    """Interactive comparison of raw, short and medium results."""
    raw_s = raw[col]
    short_s = short[col].reindex(raw_s.index)    # dropped rows -> NaN
    medium_s = medium[col].reindex(raw_s.index)
    raw_s, short_s, medium_s = (
        s.loc[start:end] for s in (raw_s, short_s, medium_s)
    )

    short_fill = raw_s.isna() & short_s.notna()
    medium_fill = short_s.isna() & medium_s.notna()

    fig = go.Figure()
    fig.add_scatter(x=raw_s.index, y=raw_s, name="raw",
                    mode="lines", line=dict(color="royalblue", width=1))
    fig.add_scatter(x=short_s.index[short_fill], y=short_s[short_fill],
                    name="short fill", mode="markers",
                    marker=dict(color="orange", size=6))
    fig.add_scatter(x=medium_s.index[medium_fill],
                    y=medium_s[medium_fill], name="medium fill",
                    mode="markers", marker=dict(color="red", size=6))
    fig.update_layout(
        title=f"{station} - {col}",
        yaxis_title=col,
        hovermode="x unified",
        template="plotly_white",
    )
    fig.update_xaxes(rangeslider_visible=True)
    return fig

In [78]:
FREQ = pd.Timedelta("15min")


# Load data
stations = ['PGB', 'PLIB', 'USAM', 'UTEC']
generalpath = "data/air/consolidated"
storepath = "data/air/processed"

for station in stations:
    stages: dict[str, tuple[pd.DataFrame, ...]] = {}

    filepath = os.path.join(generalpath, f"{station}.csv")
    df = pd.read_csv(filepath, delimiter= ";", parse_dates=['Timestamp'], index_col='Timestamp')
    raw = df.resample('15Min').mean()  # Resample to 35-minute intervals

    outliers = detect_outliers_iqr(raw)
    print(outliers.sum())                      # outliers per column
    raw = raw.mask(outliers)

    short = fill_short_gaps(raw)
    medium = fill_medium_gaps(short)
    stages[station] = (raw, short, medium)

    for station, (raw, short, medium) in stages.items():
        plot_stages_plotly(raw, short, medium, station, col="CO2")#.show()



ICA     665
TEM       3
HUM       0
CO2     245
VOC    1042
dtype: int64
ICA      0
TEM      0
HUM      0
CO2     90
VOC    234
dtype: int64
ICA    569
CO2     28
VOC    419
TEM      3
HUM      2
dtype: int64
ICA     55
TEM     24
HUM      0
CO2    108
VOC    572
dtype: int64


In [79]:
is_nan = df['CO2'].isna()
(is_nan != is_nan.shift()).cumsum()

Timestamp
2025-05-03 15:06:00+00:00    1
2025-05-03 15:16:00+00:00    1
2025-05-03 15:26:00+00:00    1
2025-05-03 15:36:00+00:00    1
2025-05-03 15:46:00+00:00    1
                            ..
2025-08-31 23:18:00+00:00    1
2025-08-31 23:28:00+00:00    1
2025-08-31 23:38:00+00:00    1
2025-08-31 23:48:00+00:00    1
2025-08-31 23:58:00+00:00    1
Name: CO2, Length: 11014, dtype: int32